**Загрузка данных**

Импортируем библиотеку Pandas:

In [2]:
import pandas as pd

In [3]:
raw_data = pd.read_csv('./titanic.csv', index_col="PassengerId")

Команды для сохранения и загрузки:
- tablename.to_csv('./filename.csv', index=None)
- tablename = pandas.read_csv('./filename.csv')

**Анализ и обработка данных**

In [4]:
len(raw_data)

891

In [6]:
raw_data.columns

Index(['Survived', 'Pclass', 'Name', 'Sex', 'Age', 'SibSp', 'Parch', 'Ticket',
       'Fare', 'Cabin', 'Embarked'],
      dtype='str')

In [7]:
raw_data['Survived']

PassengerId
1      0
2      1
3      1
4      1
5      0
      ..
887    0
888    1
889    0
890    1
891    0
Name: Survived, Length: 891, dtype: int64

In [8]:
raw_data[['Name', 'Age']]

,Name,Age
PassengerId,,
1,"Braund, Mr. Owen Harris",22.0
2,"Cumings, Mrs. John Bradley (Florence Briggs Th...",38.0
3,"Heikkinen, Miss. Laina",26.0
4,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",35.0
5,"Allen, Mr. William Henry",35.0
...,...,...
887,"Montvila, Rev. Juozas",27.0
888,"Graham, Miss. Margaret Edith",19.0
889,"Johnston, Miss. Catherine Helen ""Carrie""",NaN


In [9]:
sum(raw_data['Survived'])

342

Например, здесь отсутствуют данные:

In [10]:
raw_data['Cabin']

PassengerId
1       NaN
2       C85
3       NaN
4      C123
5       NaN
       ... 
887     NaN
888     B42
889     NaN
890    C148
891     NaN
Name: Cabin, Length: 891, dtype: str

Всего отсутствует:

In [11]:
raw_data.isna().sum()

Survived      0
Pclass        0
Name          0
Sex           0
Age         177
SibSp         0
Parch         0
Ticket        0
Fare          0
Cabin       687
Embarked      2
dtype: int64

Удалим столбец с большим количеством недостающих данных:

In [12]:
clean_data = raw_data.drop('Cabin', axis=1)

Аргументы функции drop:
- имя столбца, который хотим удалить;
- параметр axis, который равен 1, когда хотим удалить столбец, и 0, когда
строку.

In [13]:
clean_data['Age']

PassengerId
1      22.0
2      38.0
3      26.0
4      35.0
5      35.0
       ... 
887    27.0
888    19.0
889     NaN
890    26.0
891    32.0
Name: Age, Length: 891, dtype: float64

В другом столбце заменим пропуски медианным значением - числом, которое находится в середине набора, если его упорядочить по возрастанию, то есть такое число, что половина чисел из набора не меньше него, а другая половина не больше.

In [14]:
median_age = clean_data["Age"].median()
clean_data["Age"] = clean_data["Age"].fillna(median_age)

In [15]:
clean_data["Embarked"] = clean_data["Embarked"].fillna('U')

Сохраним обработанные данные:

In [16]:
clean_data.to_csv('./clean_titanic_data.csv', index=None)

**Конструирование признаков**

Преобразование категориальных данных в числовые:

In [17]:
gender_columns = pd.get_dummies(clean_data['Sex'], prefix='Sex').astype(int)
embarked_columns = pd.get_dummies(clean_data["Embarked"], prefix="Embarked").astype(int)
preprocessed_data = pd.concat([clean_data, gender_columns], axis=1)
preprocessed_data = pd.concat([preprocessed_data, embarked_columns], axis=1)
preprocessed_data = preprocessed_data.drop(['Sex', 'Embarked'], axis=1)

In [18]:
categorized_pclass_columns = pd.get_dummies(preprocessed_data['Pclass'], prefix='Pclass').astype(int)
preprocessed_data = pd.concat([preprocessed_data, categorized_pclass_columns], axis=1)
preprocessed_data = preprocessed_data.drop(['Pclass'], axis=1)

Превращение числовых данных в категориальные (биннинг):

In [19]:
bins = [0, 10, 20, 30, 40, 50, 60, 70, 80]
categorized_age = pd.cut(preprocessed_data['Age'], bins)
preprocessed_data['Categorized_age'] = categorized_age
preprocessed_data = preprocessed_data.drop(["Age"], axis=1)

In [20]:
preprocessed_data.head()

,Survived,Name,SibSp,Parch,Ticket,Fare,Sex_female,Sex_male,Embarked_C,Embarked_Q,Embarked_S,Embarked_U,Pclass_1,Pclass_2,Pclass_3,Categorized_age
PassengerId,,,,,,,,,,,,,,,,
1,0,"Braund, Mr. Owen Harris",1,0,A/5 21171,7.2500,0,1,0,0,1,0,0,0,1,"(20, 30]"
2,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",1,0,PC 17599,71.2833,1,0,1,0,0,0,1,0,0,"(30, 40]"
3,1,"Heikkinen, Miss. Laina",0,0,STON/O2. 3101282,7.9250,1,0,0,0,1,0,0,0,1,"(20, 30]"
4,1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",1,0,113803,53.1000,1,0,0,0,1,0,1,0,0,"(30, 40]"
5,0,"Allen, Mr. William Henry",0,0,373450,8.0500,0,1,0,0,1,0,0,0,1,"(30, 40]"


Отбор признаков:

In [21]:
preprocessed_data.index.name = None

In [22]:
preprocessed_data.head()

,Survived,Name,SibSp,Parch,Ticket,Fare,Sex_female,Sex_male,Embarked_C,Embarked_Q,Embarked_S,Embarked_U,Pclass_1,Pclass_2,Pclass_3,Categorized_age
1,0,"Braund, Mr. Owen Harris",1,0,A/5 21171,7.2500,0,1,0,0,1,0,0,0,1,"(20, 30]"
2,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",1,0,PC 17599,71.2833,1,0,1,0,0,0,1,0,0,"(30, 40]"
3,1,"Heikkinen, Miss. Laina",0,0,STON/O2. 3101282,7.9250,1,0,0,0,1,0,0,0,1,"(20, 30]"
4,1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",1,0,113803,53.1000,1,0,0,0,1,0,1,0,0,"(30, 40]"
5,0,"Allen, Mr. William Henry",0,0,373450,8.0500,0,1,0,0,1,0,0,0,1,"(30, 40]"


In [23]:
preprocessed_data = preprocessed_data.drop(['Name', 'Ticket'], axis=1)

In [24]:
preprocessed_data.head()

,Survived,SibSp,Parch,Fare,Sex_female,Sex_male,Embarked_C,Embarked_Q,Embarked_S,Embarked_U,Pclass_1,Pclass_2,Pclass_3,Categorized_age
1,0,1,0,7.2500,0,1,0,0,1,0,0,0,1,"(20, 30]"
2,1,1,0,71.2833,1,0,1,0,0,0,1,0,0,"(30, 40]"
3,1,0,0,7.9250,1,0,0,0,1,0,0,0,1,"(20, 30]"
4,1,1,0,53.1000,1,0,0,0,1,0,1,0,0,"(30, 40]"
5,0,0,0,8.0500,0,1,0,0,1,0,0,0,1,"(30, 40]"


Сохраним данные:

In [25]:
preprocessed_data.to_csv('./preprocessed_titanic_data.csv', index=None)